In [ ]:
import mne
import torch
import numpy  as np
from torch import nn
from moabb.datasets import BNCI2014_001
from braindecode.models import ShallowFBCSPNet
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.model_selection import train_test_split
import copy
pio.renderers.default = "vscode"

In [10]:
# used model and dataset
dataset = BNCI2014_001()
data = dataset.get_data()
model = ShallowFBCSPNet(n_chans=22, n_outputs=4, n_times=576)

/home/machlearn/IC/venv/lib/python3.14/site-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
/home/machlearn/IC/venv/lib/python3.14/site-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
/home/machlearn/IC/venv/lib/python3.14/site-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
/home/machlearn/IC/venv/lib/python3.14/site-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_m

In [17]:
# defined loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model.parameters(), 
                            lr=0.01)

In [ ]:
# WITHIN-SUBJECT 

In [ ]:
subjects = list(data.keys())

def epochs_from_session(session_dict, tmin=0, tmax=4.0):

    raws = list(session_dict.values())
    raw = mne.concatenate_raws(raws) if len(raws) > 1 else raws[0]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=tmin, tmax=tmax, baseline=None, preload=True)
    epochs.pick('eeg')
    return epochs


# dicionário: subject_id -> {"train": epochs_sessao0, "test": epochs_sessao1}
within_subject_tensors = {}

for subj in subjects:
    session_keys = list(data[subj].keys())

    train_epochs = epochs_from_session(data[subj][session_keys[0]])
    test_epochs = epochs_from_session(data[subj][session_keys[1]])

    X_tr = train_epochs.get_data() * 1e6
    y_tr = train_epochs.events[:, -1]
    X_te = test_epochs.get_data() * 1e6
    y_te = test_epochs.events[:, -1]

    min_lbl = y_tr.min()

    within_subject_tensors[subj] = {
        "X_train": torch.tensor(X_tr, dtype=torch.float32),
        "y_train": torch.tensor(y_tr - min_lbl, dtype=torch.long),
        "X_test": torch.tensor(X_te, dtype=torch.float32),
        "y_test": torch.tensor(y_te - min_lbl, dtype=torch.long),
    }

n_epochs = 1000
results_by_subject = {}
for subj, tensors in within_subject_tensors.items():
    X_train_sub = tensors["X_train"]
    y_train_sub = tensors["y_train"]
    X_test_sub = tensors["X_test"]
    y_test_sub = tensors["y_test"]

    n_times = X_train_sub.shape[-1]
    
    model = ShallowFBCSPNet(n_chans=22, n_outputs=4, n_times=n_times)
    optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)
   
    train_loss_value_within = [] # keep track of the loss values
    test_loss_value_within = []
    epoch_count_within = []

    for epoch in range(n_epochs):
        model.train() 
        #forward pass
        y_pred = model(X_train_sub)

        #calculate the loss
        loss = loss_fn(y_pred, y_train_sub)

        #zero grad optimizer
        optimizer.zero_grad()

        #loss backward
        loss.backward()

        #step optimizerS
        optimizer.step()

        #testing

        model.eval()

        with torch.inference_mode():
            test_pred = model(X_test_sub)
            test_loss = loss_fn(test_pred, y_test_sub)

        if epoch % 100 == 0:
            epoch_count_within.append(epoch)
            train_loss_value_within.append(loss.detach().numpy())
            test_loss_value_within.append(test_loss.detach().numpy())
            print(f"Epoch: {epoch} | Train loss: {loss} | Test loss: {test_loss}")

    model.eval()
    with torch.inference_mode():
        final_train_logits = model(X_train_sub)
        final_test_logits = model(X_test_sub)

    results_by_subject[subj] = {
        "y_train_true_within": y_train_sub.numpy(),
        "y_train_pred_within": final_train_logits.argmax(dim=1).numpy(),
        "y_test_true_within": y_test_sub.numpy(),
        "y_test_pred_within": final_test_logits.argmax(dim=1).numpy(),
        "epoch_count_within": epoch_count_within,
        "train_loss_value_within": train_loss_value_within,
        "test_loss_value_within": test_loss_value_within,
    }

    print(f"Finished subject {subj}")

In [ ]:
def plot_expected_vs_predicted(results_by_subject):
    subjects = list(results_by_subject.keys())
    fig = go.Figure()

    traces_per_subject = 4  # train_true, train_pred, test_true, test_pred

    for i, subj in enumerate(subjects):
        r = results_by_subject[subj]
        n_train = len(r["y_train_true_within"])
        n_test = len(r["y_test_true_within"])

        visible = (i == 0) 

        fig.add_trace(go.Scatter(
            x=np.arange(n_train), y=r["y_train_true_within"],
            mode="markers", marker=dict(color="blue", size=6, symbol="circle"),
            name="Train - esperado", visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=np.arange(n_train), y=r["y_train_pred_within"],
            mode="markers", marker=dict(color="lightblue", size=6, symbol="x"),
            name="Train - obtido", visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=np.arange(n_test), y=r["y_test_true_within"],
            mode="markers", marker=dict(color="green", size=6, symbol="circle"),
            name="Test - esperado", visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=np.arange(n_test), y=r["y_test_pred_within"],
            mode="markers", marker=dict(color="red", size=6, symbol="x"),
            name="Test - obtido", visible=visible
        ))

    # dropdown
    buttons = []
    for i, subj in enumerate(subjects):
        visibility = [False] * (len(subjects) * traces_per_subject)
        start = i * traces_per_subject
        for j in range(traces_per_subject):
            visibility[start + j] = True

        buttons.append(dict(
            label=f"Sujeito {subj}",
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Esperado vs Obtido — Sujeito {subj}"}]
        ))

    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=1.15, y=1.0, xanchor="left", yanchor="top"
        )],
        width=900,
        height=600,
        title=f"Esperado vs Obtido — Sujeito {subjects[0]}",
        xaxis_title="Índice da amostra (epoch de EEG)",
        yaxis_title="Classe prevista",
        legend=dict(x=0.02, y=0.98)
    )

    fig.show()

plot_expected_vs_predicted(results_by_subject)

In [ ]:
def plot_loss_curve(results_by_subject):
    subjects = list(results_by_subject.keys())
    fig = go.Figure()

    traces_per_subject = 2  # train_loss, val_loss

    for i, subj in enumerate(subjects):
        r = results_by_subject[subj]
        visible = (i == 0)

        fig.add_trace(go.Scatter(
            x=r["epoch_count_within"],
            y=r["train_loss_values_within"],
            mode="lines+markers",
            marker=dict(color="blue", size=4),
            name="Train Loss",
            visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=r["epoch_count_within"],
            y=r["test_loss_values_within"],  
            mode="lines+markers",
            marker=dict(color="red", size=4),
            name="Test Loss",              
            visible=visible
        ))

    buttons = []
    for i, subj in enumerate(subjects):
        visibility = [False] * (len(subjects) * traces_per_subject)
        start = i * traces_per_subject
        for j in range(traces_per_subject):
            visibility[start + j] = True

        buttons.append(dict(
            label=f"Sujeito {subj}",
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Train x Test Loss — Sujeito {subj}"}]
        ))

    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=1.15, y=1.0, xanchor="left", yanchor="top"
        )],
        width=900,
        height=600,
        title=f"Train x Test Loss — Sujeito {subjects[0]}",
        xaxis_title="Epochs",
        yaxis_title="Loss",
        legend=dict(x=0.02, y=0.98)
    )

    fig.show()

plot_loss_curve(results_by_subject)

In [ ]:
#WITHIN-SUBJECT WITH VALIDATION

In [ ]:
subjects = list(data.keys())

def epochs_from_session(session_dict, tmin=0, tmax=4.0):
    raws = list(session_dict.values())
    raw = mne.concatenate_raws(raws) if len(raws) > 1 else raws[0]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=tmin, tmax=tmax, baseline=None, preload=True)
    epochs.pick('eeg')
    return epochs

subject_train_epochs = {}
subject_test_epochs = {}

for subj in subjects:
    session_keys = list(data[subj].keys())
    subject_train_epochs[subj] = epochs_from_session(data[subj][session_keys[0]])
    subject_test_epochs[subj]  = epochs_from_session(data[subj][session_keys[1]])

all_events = np.concatenate(
    [subject_train_epochs[s].events[:, -1] for s in subjects] +
    [subject_test_epochs[s].events[:, -1] for s in subjects]
)
min_lbl = all_events.min()

hybrid_tensors = {}

for target_subj in subjects:
    other_subjs = [s for s in subjects if s != target_subj]


    X_others_list, y_others_list = [], []
    for s in other_subjs:
        ep = subject_train_epochs[s]
        X_others_list.append(ep.get_data() * 1e6)
        y_others_list.append(ep.events[:, -1] - min_lbl)

    X_others = np.concatenate(X_others_list, axis=0)
    y_others = np.concatenate(y_others_list, axis=0)


    ep_target = subject_train_epochs[target_subj]
    X_target = ep_target.get_data() * 1e6
    y_target = ep_target.events[:, -1] - min_lbl

    X_target_train, X_target_val, y_target_train, y_target_val = train_test_split(
        X_target, y_target,
        test_size=0.5,
        stratify=y_target,
        random_state=42
    )

    # treino final = outros sujeitos inteiros + metade do sujeito alvo
    X_train = np.concatenate([X_others, X_target_train], axis=0)
    y_train = np.concatenate([y_others, y_target_train], axis=0)

    # validação = outra metade do sujeito alvo
    X_val = X_target_val
    y_val = y_target_val

    # teste = sessão de teste do sujeito alvo (within-subject de verdade)
    ep_test = subject_test_epochs[target_subj]
    X_test = ep_test.get_data() * 1e6
    y_test = ep_test.events[:, -1] - min_lbl

    hybrid_tensors[target_subj] = {
        "X_train": torch.tensor(X_train, dtype=torch.float32),
        "y_train": torch.tensor(y_train, dtype=torch.long),
        "X_val":   torch.tensor(X_val, dtype=torch.float32),
        "y_val":   torch.tensor(y_val, dtype=torch.long),
        "X_test":  torch.tensor(X_test, dtype=torch.float32),
        "y_test":  torch.tensor(y_test, dtype=torch.long),
    }

n_epochs = 1000
patience = 50
results_by_subject = {}

for subj, tensors in hybrid_tensors.items():
    X_train_sub = tensors["X_train"]
    y_train_sub = tensors["y_train"]
    X_val_sub   = tensors["X_val"]
    y_val_sub   = tensors["y_val"]
    X_test_sub  = tensors["X_test"]
    y_test_sub  = tensors["y_test"]

    n_times = X_train_sub.shape[-1]
    model = ShallowFBCSPNet(n_chans=22, n_outputs=4, n_times=n_times)
    optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)

    train_loss_values_validation, test_loss_values_validation, epoch_count_validation = [], [], []
    best_val_loss = float("inf")
    best_model_state = None
    epochs_no_improve = 0

    for epoch in range(n_epochs):
        model.train() 
        #forward pass
        y_pred = model(X_train_sub)

        #calculate the loss
        loss = loss_fn(y_pred, y_train_sub)

        #zero grad optimizer
        optimizer.zero_grad()

        #loss backward
        loss.backward()

        #step optimizerS
        optimizer.step()

        #testing

        model.eval()
        with torch.inference_mode():
            val_pred = model(X_val_sub)
            val_loss = loss_fn(val_pred, y_val_sub)

        if epoch % 10 == 0:
            epoch_count_validation.append(epoch)
            train_loss_values_validation.append(loss.detach().numpy())
            test_loss_values_validation.append(val_loss.detach().numpy())

        if epoch % 100 == 0:
            print(f"Subj {subj} | Epoch: {epoch} | Train loss: {loss:.4f} | Val loss: {val_loss:.4f}")

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Subj {subj}: early stopping na época {epoch}")
                break

    model.load_state_dict(best_model_state)
    model.eval()
    with torch.inference_mode():
        final_train_logits = model(X_train_sub)
        final_val_logits = model(X_val_sub)
        final_test_logits = model(X_test_sub)
        final_test_loss = loss_fn(final_test_logits, y_test_sub)

    results_by_subject[subj] = {
        "y_train_true_validation": y_train_sub.numpy(),
        "y_train_pred_validation": final_train_logits.argmax(dim=1).numpy(),
        "y_val_true_validation": y_val_sub.numpy(),
        "y_val_pred_validation": final_val_logits.argmax(dim=1).numpy(),
        "y_test_true_validation": y_test_sub.numpy(),
        "y_test_pred_validation": final_test_logits.argmax(dim=1).numpy(),
        "epoch_count_validation": epoch_count_validation,
        "train_loss_values_validation": train_loss_values_validation,
        "val_loss_values_validation": test_loss_values_validation,
        "final_test_loss_validation": final_test_loss.item(),
    }

    print(f"Finished subject {subj} | Best val loss: {best_val_loss:.4f} | Test loss: {final_test_loss:.4f}")

In [ ]:
def plot_expected_vs_predicted(results_by_subject):
    subjects = list(results_by_subject.keys())
    fig = go.Figure()

    traces_per_subject = 4  # train_true, train_pred, test_true, test_pred

    for i, subj in enumerate(subjects):
        r = results_by_subject[subj]
        n_train = len(r["y_train_true_validation"])
        n_test = len(r["y_test_true_validation"])

        visible = (i == 0) 

        fig.add_trace(go.Scatter(
            x=np.arange(n_train), y=r["y_train_true_validation"],
            mode="markers", marker=dict(color="blue", size=6, symbol="circle"),
            name="Train - esperado", visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=np.arange(n_train), y=r["y_train_pred_validation"],
            mode="markers", marker=dict(color="lightblue", size=6, symbol="x"),
            name="Train - obtido", visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=np.arange(n_test), y=r["y_test_true_validation"],
            mode="markers", marker=dict(color="green", size=6, symbol="circle"),
            name="Test - esperado", visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=np.arange(n_test), y=r["y_test_pred_validation"],
            mode="markers", marker=dict(color="red", size=6, symbol="x"),
            name="Test - obtido", visible=visible
        ))

    # dropdown
    buttons = []
    for i, subj in enumerate(subjects):
        visibility = [False] * (len(subjects) * traces_per_subject)
        start = i * traces_per_subject
        for j in range(traces_per_subject):
            visibility[start + j] = True

        buttons.append(dict(
            label=f"Sujeito {subj}",
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Esperado vs Obtido — Sujeito {subj}"}]
        ))

    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=1.15, y=1.0, xanchor="left", yanchor="top"
        )],
        width=900,
        height=600,
        title=f"Esperado vs Obtido — Sujeito {subjects[0]}",
        xaxis_title="Índice da amostra (epoch de EEG)",
        yaxis_title="Classe prevista",
        legend=dict(x=0.02, y=0.98)
    )

    fig.show()

plot_expected_vs_predicted(results_by_subject)

In [ ]:
def plot_loss_curve(results_by_subject):
    subjects = list(results_by_subject.keys())
    fig = go.Figure()

    traces_per_subject = 2  # train_loss, val_loss

    for i, subj in enumerate(subjects):
        r = results_by_subject[subj]
        visible = (i == 0)

        fig.add_trace(go.Scatter(
            x=r["epoch_count_validation"],
            y=r["train_loss_values_validation"],
            mode="lines+markers",
            marker=dict(color="blue", size=4),
            name="Train Loss",
            visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=r["epoch_count_validation"],
            y=r["val_loss_values_validation"],  
            mode="lines+markers",
            marker=dict(color="red", size=4),
            name="Validation Loss",              
            visible=visible
        ))

    buttons = []
    for i, subj in enumerate(subjects):
        visibility = [False] * (len(subjects) * traces_per_subject)
        start = i * traces_per_subject
        for j in range(traces_per_subject):
            visibility[start + j] = True

        buttons.append(dict(
            label=f"Sujeito {subj}",
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Train x Validation Loss — Sujeito {subj}"}]
        ))

    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=1.15, y=1.0, xanchor="left", yanchor="top"
        )],
        width=900,
        height=600,
        title=f"Train x Validation Loss — Sujeito {subjects[0]}",
        xaxis_title="Epochs",
        yaxis_title="Loss",
        legend=dict(x=0.02, y=0.98)
    )

    fig.show()

plot_loss_curve(results_by_subject)

In [ ]:
# CROSS-SUBJECT 

In [ ]:

subjects = list(data.keys())

def epochs_from_subject(session_dict, tmin=0, tmax=4.0):

    raws = []
    for sess in session_dict.values():
        raws.extend(list(sess.values()))

    raw = mne.concatenate_raws(raws) if len(raws) > 1 else raws[0]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=tmin, tmax=tmax, baseline=None, preload=True)
    epochs.pick('eeg')
    return epochs

subject_epochs = {}
for subj in subjects:
    subject_epochs[subj] = epochs_from_subject(data[subj])


all_events = np.concatenate([subject_epochs[s].events[:, -1] for s in subjects])
min_lbl = all_events.min()

cross_subject_tensor = {}
for test_subj in subjects:
    train_subjs = [s for s in subjects if s != test_subj]

    X_train_list, y_train_list = [], []
    for s in train_subjs:
        ep = subject_epochs[s]
        X_train_list.append(ep.get_data() * 1e6)
        y_train_list.append(ep.events[:, -1])

    X_train = np.concatenate(X_train_list, axis=0)
    y_train = np.concatenate(y_train_list, axis=0)

    X_test = subject_epochs[test_subj].get_data() * 1e6
    y_test = subject_epochs[test_subj].events[:, -1]

    cross_subject_tensor[test_subj] = {
        "X_train": torch.tensor(X_train, dtype=torch.float32),
        "y_train": torch.tensor(y_train - min_lbl, dtype=torch.long),
        "X_test": torch.tensor(X_test, dtype=torch.float32),
        "y_test": torch.tensor(y_test - min_lbl, dtype=torch.long),
    }


n_epochs = 1000
results_by_subject = {}
for subj, tensors in cross_subject_tensor.items():
    X_train_sub = tensors["X_train"]
    y_train_sub = tensors["y_train"]
    X_test_sub = tensors["X_test"]
    y_test_sub = tensors["y_test"]

    n_times = X_train_sub.shape[-1]
    
    model = ShallowFBCSPNet(n_chans=22, n_outputs=4, n_times=n_times)
    optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)
   
    train_loss_values_cross = [] # keep track of the loss values
    test_loss_values_cross = []
    epoch_count_cross = []

    for epoch in range(n_epochs):
        model.train() 
        #forward pass
        y_pred = model(X_train_sub)

        #calculate the loss
        loss = loss_fn(y_pred, y_train_sub)

        #zero grad optimizer
        optimizer.zero_grad()

        #loss backward
        loss.backward()

        #step optimizerS
        optimizer.step()

        #testing

        model.eval()

        with torch.inference_mode():
            test_pred = model(X_test_sub)
            test_loss = loss_fn(test_pred, y_test_sub)

        if epoch % 100 == 0:
            epoch_count_cross.append(epoch)
            train_loss_values_cross.append(loss.detach().numpy())
            test_loss_values_cross.append(test_loss.detach().numpy())
            print(f"Epoch: {epoch} | Train loss: {loss} | Test loss: {test_loss}")

    model.eval()
    with torch.inference_mode():
        final_train_logits = model(X_train_sub)
        final_test_logits = model(X_test_sub)

    results_by_subject[subj] = {
        "y_train_true_cross": y_train_sub.numpy(),
        "y_train_pred_cross": final_train_logits.argmax(dim=1).numpy(),
        "y_test_true_cross": y_test_sub.numpy(),
        "y_test_pred_cross": final_test_logits.argmax(dim=1).numpy(),
        "epoch_count_cross": epoch_count_cross,              
        "train_loss_values_cross": train_loss_values_cross,   
        "test_loss_values_cross": test_loss_values_cross,
    }

    print(f"Finished subject {subj}")

In [ ]:
def plot_expected_vs_predicted(results_by_subject):
    subjects = list(results_by_subject.keys())
    fig = go.Figure()

    traces_per_subject = 4  # train_true, train_pred, test_true, test_pred

    for i, subj in enumerate(subjects):
        r = results_by_subject[subj]
        n_train = len(r["y_train_true_cross"])
        n_test = len(r["y_test_true_cross"])

        visible = (i == 0) 

        fig.add_trace(go.Scatter(
            x=np.arange(n_train), y=r["y_train_true_cross"],
            mode="markers", marker=dict(color="blue", size=6, symbol="circle"),
            name="Train - esperado", visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=np.arange(n_train), y=r["y_train_pred_cross"],
            mode="markers", marker=dict(color="lightblue", size=6, symbol="x"),
            name="Train - obtido", visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=np.arange(n_test), y=r["y_test_true_cross"],
            mode="markers", marker=dict(color="green", size=6, symbol="circle"),
            name="Test - esperado", visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=np.arange(n_test), y=r["y_test_pred_cross"],
            mode="markers", marker=dict(color="red", size=6, symbol="x"),
            name="Test - obtido", visible=visible
        ))

    # dropdown
    buttons = []
    for i, subj in enumerate(subjects):
        visibility = [False] * (len(subjects) * traces_per_subject)
        start = i * traces_per_subject
        for j in range(traces_per_subject):
            visibility[start + j] = True

        buttons.append(dict(
            label=f"Sujeito {subj}",
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Esperado vs Obtido — Sujeito {subj}"}]
        ))

    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=1.15, y=1.0, xanchor="left", yanchor="top"
        )],
        width=900,
        height=600,
        title=f"Esperado vs Obtido — Sujeito {subjects[0]}",
        xaxis_title="Índice da amostra (epoch de EEG)",
        yaxis_title="Classe prevista",
        legend=dict(x=0.02, y=0.98)
    )

    fig.show()

plot_expected_vs_predicted(results_by_subject)

In [ ]:
def plot_loss_curve(results_by_subject):
    subjects = list(results_by_subject.keys())
    fig = go.Figure()

    traces_per_subject = 2  # train_loss, val_loss

    for i, subj in enumerate(subjects):
        r = results_by_subject[subj]
        visible = (i == 0)

        fig.add_trace(go.Scatter(
            x=r["epoch_count_cross"],
            y=r["train_loss_values_cross"],
            mode="lines+markers",
            marker=dict(color="blue", size=4),
            name="Train Loss",
            visible=visible
        ))
        fig.add_trace(go.Scatter(
            x=r["epoch_count_cross"],
            y=r["test_loss_values_cross"],   
            mode="lines+markers",
            marker=dict(color="red", size=4),
            name="Test Loss",             
            visible=visible
        ))

    buttons = []
    for i, subj in enumerate(subjects):
        visibility = [False] * (len(subjects) * traces_per_subject)
        start = i * traces_per_subject
        for j in range(traces_per_subject):
            visibility[start + j] = True

        buttons.append(dict(
            label=f"Sujeito {subj}",
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Train x Test Loss — Sujeito {subj}"}]
        ))

    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=1.15, y=1.0, xanchor="left", yanchor="top"
        )],
        width=900,
        height=600,
        title=f"Train x Test Loss — Sujeito {subjects[0]}",
        xaxis_title="Epochs",
        yaxis_title="Loss",
        legend=dict(x=0.02, y=0.98)
    )

    fig.show()

plot_loss_curve(results_by_subject)